# Replace PyTorch Convolutions With Microsoft MX Layers

This notebook shows the workflow you asked for:

1. take a PyTorch model as input
2. recursively replace its convolution layers with Microsoft `microxcaling` MX convolution layers
3. run a small usage example

Microsoft repo: https://github.com/microsoft/microxcaling

## Install Dependencies

Run this outside the notebook if `mx` is not importable yet:

```bash
git clone https://github.com/microsoft/microxcaling.git
cd microxcaling
pip install -e .
```

The Microsoft package expects a PyTorch/CUDA environment for the MX kernels. For CPU-only experimentation, set `custom_cuda` to `False` in the MX specs below.

In [ ]:
from copy import deepcopy

import torch
import torch.nn as nn

from mx import Conv1d, Conv2d, Conv3d, finalize_mx_specs

## MX Configuration

This config uses 8-bit shared scales, FP6 elements for weights and activations, and a block size of 32. Those are the knobs you usually change when exploring different MX formats.

In [ ]:
mx_specs = finalize_mx_specs({
    "scale_bits": 8,
    "w_elem_format": "fp6_e3m2",
    "a_elem_format": "fp6_e3m2",
    "block_size": 32,
    "bfloat": 16,
    "custom_cuda": torch.cuda.is_available(),
    "quantize_backprop": False,
})

## Replacement Function

`replace_conv_layers_with_mx` walks through any `nn.Module` tree. Whenever it finds `nn.Conv1d`, `nn.Conv2d`, or `nn.Conv3d`, it builds the matching MX layer, copies the existing weights and bias, and installs the MX layer in the same position.

In [ ]:
CONV_REPLACEMENTS = {
    nn.Conv1d: Conv1d,
    nn.Conv2d: Conv2d,
    nn.Conv3d: Conv3d,
}


def copy_conv_parameters(source, target):
    target.to(device=source.weight.device, dtype=source.weight.dtype)
    with torch.no_grad():
        target.weight.copy_(source.weight)
        target.weight.requires_grad_(source.weight.requires_grad)
        if source.bias is not None and target.bias is not None:
            target.bias.copy_(source.bias)
            target.bias.requires_grad_(source.bias.requires_grad)


def build_mx_conv(source, mx_specs, name):
    if getattr(source, "padding_mode", "zeros") != "zeros":
        raise ValueError(
            "microxcaling Conv layers do not expose PyTorch's non-zero "
            f"padding_mode behavior; layer {name!r} uses "
            f"padding_mode={source.padding_mode!r}."
        )

    mx_cls = CONV_REPLACEMENTS[type(source)]
    target = mx_cls(
        in_channels=source.in_channels,
        out_channels=source.out_channels,
        kernel_size=source.kernel_size,
        stride=source.stride,
        padding=source.padding,
        dilation=source.dilation,
        groups=source.groups,
        bias=source.bias is not None,
        mx_specs=mx_specs,
        name=name,
    )
    copy_conv_parameters(source, target)
    target.train(source.training)
    return target


def replace_conv_layers_with_mx(model, mx_specs, inplace=False):
    converted = model if inplace else deepcopy(model)

    for module_name, child in list(converted.named_children()):
        if type(child) in CONV_REPLACEMENTS:
            setattr(converted, module_name, build_mx_conv(child, mx_specs, module_name))
        else:
            replace_conv_layers_with_mx(child, mx_specs, inplace=True)

    return converted

## Example Model Input

This toy CNN stands in for any model you already have. The replacement function does not depend on this exact architecture.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 16 * 16, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = TinyCNN().eval()
print(model)

## Replace Convolution Layers

The linear layer remains unchanged. Only convolution layers are replaced.

In [ ]:
mx_model = replace_conv_layers_with_mx(model, mx_specs, inplace=False).eval()

for name, module in mx_model.named_modules():
    if isinstance(module, (Conv1d, Conv2d, Conv3d, nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        print(f"{name}: {module.__class__.__module__}.{module.__class__.__name__}")

## Run The Converted Model

The output shape should match the original model. Values may differ because MX layers quantize activations and weights according to `mx_specs`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
mx_model = mx_model.to(device)
sample = torch.randn(4, 3, 32, 32, device=device)

with torch.no_grad():
    baseline_output = model(sample)
    mx_output = mx_model(sample)

print("Baseline output shape:", tuple(baseline_output.shape))
print("MX output shape:", tuple(mx_output.shape))
print("Mean absolute difference:", (baseline_output - mx_output).abs().mean().item())

## Reuse From A Python File

The same helper is available in `mx_conv_replacement.py`:

```python
from mx_conv_replacement import make_mx_specs, replace_conv_layers_with_mx

mx_specs = make_mx_specs({"custom_cuda": True})
mx_model = replace_conv_layers_with_mx(model, mx_specs)
```

Pass your existing model as `model`; every `nn.Conv1d`, `nn.Conv2d`, and `nn.Conv3d` module inside it will be replaced.